# Simulando Dados Sensíveis para Preservar dos Dados dos Participantes do PBA

Com o objetivo de publicar os códigos utilizados para as análises no Programa Brasil Alfabetizado (PBA) no GitHub gerando conformidade com a Lei Geral de Proteção de Dados (LGPD), bem como com as regras de complience da Fundação Getulio Vargas, o presente código foi criado para simular dados sensíveis de participantes do programa PBA, sem prejudicar o resultado final das análises. Assim, sempre que uma análise for iniciada, o código a seguir deve ser rodado antes para que os dados sensíveis sejam preservados e a publicação em repositório público seja possível.

## Guia de Uso: Script de Anonimização Persistente de Dados

Este documento explica o funcionamento e como utilizar o script de automação em Python para anonimização de dados pessoais (PII). O script foi desenvolvido para garantir a conformidade com a **LGPD (Lei Geral de Proteção de Dados)** antes da publicação de repositórios no GitHub.

### 🎯 Objetivo do Script

O script lê arquivos nas extensões `.xlsx` (Excel) e `.csv`, identifica colunas contendo nomes e CPFs reais e substitui esses valores por dados fictícios válidos gerados pela biblioteca `Faker`. 

#### Diferenciais desta solução:
* **Consistência Referencial (De-Para):** Se uma pessoa aparece em múltiplos arquivos (ex: como Alfabetizadora em um arquivo e Coordenadora em outro), ela receberá exatamente o mesmo nome e CPF falsos em todas as tabelas.
* **Persistência de Estado:** Os mapeamentos são salvos localmente em arquivos de dicionário (`.csv`). Se você rodar o script hoje e receber novos arquivos no próximo mês, o script lembrará dos nomes fictícios já atribuídos.
* **Preservação de Dados Nulos:** Caso um participante não possua CPF em um determinado momento, a célula continuará vazia (`NaN`), sem gerar dados falsos para registros inexistentes.

---

### 🛠️ Pré-requisitos

Antes de executar o script, certifique-se de ter o Python instalado e as bibliotecas necessárias. Você pode instalá-las executando o comando abaixo no terminal:

```bash
pip install pandas faker openpyxl
```
- *Nota: A biblioteca `openpyxl` é necessária para o Pandas conseguir ler e escrever arquivos do Excel (`.xlsx`).*

---

### 📂 Estrutura de Pastas Recomendada
Para que o script funcione sem modificações nos caminhos, organize a raiz do seu projeto da seguinte forma:

```bash
seu_projeto/
│
├── anonimizar_dados.py          # O código deste script
├── .gitignore                   # Arquivo de proteção do Git
│
└── Data_files/
    ├── original_data/           # ⚠️ COLOQUE SEUS ARQUIVOS SENSÍVEIS AQUI
    │   ├── turmas_2025.xlsx
    │   └── cadastros.csv
    │
    └── public_data/             # ✨ OS ARQUIVOS LIMPOS SERÃO GERADOS AQUI
```

---

### ⚙️ Como Configurar e Customizar
Abra o arquivo do script e verifique os dois pontos de atenção principais:

#### Nomes das Colunas (Linha 47)
O script varre apenas as colunas explicitamente listadas nas variáveis `colunas_de_pessoas` e `colunas_cpf`, a primeira deve conter colunas com nomes de pessoas e a segunda com CPFs. Caso sua base real utilize cabeçalhos diferentes, adicione-os à lista (o script diferencia maiúsculas de minúsculas):

```bash
colunas_de_pessoas = ['Nome', 'Alfabetizador', 'ALFABETIZADOR', 'Coordenador', 'NOME_DO_ALUNO']
colunas_de_cpf = ['CPF']
```

---

### 🚀 Como Executar
1. Insira todos os arquivos que deseja tratar dentro da pasta `Data_files/original_data/`.

2. Execute o script no Jupyter Notebook.

**O que acontece após a execução?**

O script criará (caso não existam) dois arquivos na raiz do projeto: `dicionario_nomes.csv` e `dicionario_cpfs.csv`. Eles guardam a tabela de tradução "Dado Real ➔ Dado Falso".

Os arquivos tratados serão gravados na pasta `Data_files/public_data/` com os mesmos nomes originais. Estes são os arquivos seguros para usar no GitHub.

---

### 🔒 Segurança Crítica (`.gitignore`)
**IMPORTANTE:** Os arquivos originais e os dicionários criados contêm os dados sensíveis cruzados com os falsos. Eles NUNCA podem ser enviados para o GitHub.

Crie um arquivo chamado .gitignore na raiz do seu projeto e adicione as seguintes linhas:

```bash
# Ignorar pasta com dados sensíveis originais
Data_files/original_data/

# Ignorar dicionários de tradução (chaves de anonimização)
dicionario_nomes.csv
dicionario_cpfs.csv

# Ignorar arquivos temporários do sistema e do Excel
~$*.xlsx
.ipynb_checkpoints/
```

---

### 🔄 Como lidar com correções manuais de dados?
Se na base original o nome de um coordenador foi corrigido devido a um erro de digitação (ex: de "Mariano Sila" para "Mariano Silva"), o script assumirá inicialmente que são pessoas diferentes e gerará dois nomes falsos.

Para corrigir e unificar (se necessário):

1. Abra o arquivo dicionario_nomes.csv criado na raiz do seu projeto.

2. Localize a linha do nome corrigido.

3. Altere manualmente a coluna Falso para que ambas as variações do nome real apontem para o mesmo nome fictício.

4. Salve o CSV e reexecute o script.


In [6]:
import os
import pandas as pd
from faker import Faker

# 1. Configurações Iniciais
fake = Faker('pt_BR')

# ==============================================================================
# CONFIGURAÇÃO DE ANONIMIZAÇÃO: Centralize aqui todas as colunas que deseja proteger
# ==============================================================================

CONFIG = {
    'nomes': {
        'colunas': ['Nome', 'Alfabetizador', 'ALFABETIZADOR', 'Coordenador'],
        'gerador': fake.name,
        'arquivo': 'dicionario_nomes.csv'
    },
    'cpfs': {
        'colunas': ['CPF'],
        'gerador': fake.cpf,
        'arquivo': 'dicionario_cpfs.csv'
    },
    'telefones': {
        'colunas': ['Telefone'],
        'gerador': fake.phone_number,
        'arquivo': 'dicionario_telefones.csv'
    },
    'datas_nasc': {
        'colunas': ['Data de nascimento'],
        # Gera uma data fictícia formatada como DD/MM/AAAA para manter o padrão do Excel
        'gerador': lambda: fake.date_of_birth(minimum_age=15, maximum_age=90).strftime('%d/%m/%Y'),
        'arquivo': 'dicionario_datas.csv'
    },
    'bairros': {
        'colunas': ['Bairro'],
        'gerador': fake.bairro,
        'arquivo': 'dicionario_bairros.csv'
    },
    'enderecos': {
        'colunas': ['Endereço completo'],
        'gerador': fake.address,
        'arquivo': 'dicionario_enderecos.csv'
    }
}


In [7]:
# Definindo caminhos para os dados
pasta_origem = "Data_files/original_data/"
pasta_destino = "Data_files/public_data/"
os.makedirs(pasta_destino, exist_ok=True)

# 2. Carregar todos os dicionários existentes para a memória de forma dinâmica
map_globais = {}
for grupo, info in CONFIG.items():
    if os.path.exists(info['arquivo']):
        df_dict = pd.read_csv(info['arquivo'], dtype=str)
        map_globais[grupo] = dict(zip(df_dict['Real'], df_dict['Falso']))
    else:
        map_globais[grupo] = {}

# 3. Função única e dinâmica para atualizar os mapeamentos
def atualizar_dicionarios_dinamico(df):
    for grupo, info in CONFIG.items():
        for col in info['colunas']:
            if col in df.columns:
                # Extrai valores reais únicos ignorando os nulos
                valores_atuais = df[col].dropna().astype(str).unique()
                for val in valores_atuais:
                    if val not in map_globais[grupo]:
                        # Executa a função geradora correspondente do Faker
                        map_globais[grupo][val] = info['gerador']()

# 4. Função para salvar todos os dicionários atualizados de volta no disco
def salvar_dicionarios_dinamico():
    for grupo, info in CONFIG.items():
        df_salvar = pd.DataFrame(list(map_globais[grupo].items()), columns=['Real', 'Falso'])
        df_salvar.to_csv(info['arquivo'], index=False)

# 5. Loop Principal: Ler, Anonimizar e Salvar
arquivos = [f for f in os.listdir(pasta_origem) if f.lower().endswith(('.xlsx', '.csv'))]

for nome_arquivo in arquivos:
    caminho_completo = os.path.join(pasta_origem, nome_arquivo)
    extensao = os.path.splitext(nome_arquivo)[1].lower()
    
    # A: Leitura dinâmica baseada na extensão
    if extensao == '.xlsx':
        df = pd.read_excel(caminho_completo, dtype=str) 
    elif extensao == '.csv':
        df = pd.read_csv(caminho_completo, dtype=str)
    
    # Passo B: Descobrir dados novos e atualizar os dicionários na memória
    atualizar_dicionarios_dinamico(df)
    
    # Passo C: Aplicar a tradução no DataFrame para cada grupo configurado
    for grupo, info in CONFIG.items():
        for col in info['colunas']:
            if col in df.columns:
                df[col] = df[col].astype(str).map(map_globais[grupo])
            
    # D: Salvamento dinâmico baseado na extensão
    if extensao == '.xlsx':
        df.to_excel(os.path.join(pasta_destino, nome_arquivo), index=False)
    elif extensao == '.csv':
        df.to_csv(os.path.join(pasta_destino, nome_arquivo), index=False, encoding='utf-8')
        
    print(f"✅ {nome_arquivo} processado e salvo na pasta pública.")

# Salva as atualizações de todos os arquivos de mapeamento de uma vez só
salvar_dicionarios_dinamico()
print("🔒 Todos os dicionários persistentes foram atualizados e salvos com sucesso!")


✅ cadastro_alfabetizandos_06022026.xlsx processado e salvo na pasta pública.
✅ porcentagem_diag_nivel_municipio.xlsx processado e salvo na pasta pública.
✅ relatorios_turmas_diagnostica_02122025.xlsx processado e salvo na pasta pública.
✅ relatorios_turmas_diagnostica_03112025.xlsx processado e salvo na pasta pública.
✅ relatorios_turmas_diagnostica_04122025.xlsx processado e salvo na pasta pública.
✅ relatorios_turmas_diagnostica_11092025.xlsx processado e salvo na pasta pública.
✅ relatorios_turmas_diagnostica_12062025.xlsx processado e salvo na pasta pública.
✅ relatorios_turmas_diagnostica_12092025.xlsx processado e salvo na pasta pública.
✅ relatorios_turmas_diagnostica_17112025.xlsx processado e salvo na pasta pública.
✅ relatorios_turmas_diagnostica_18062025.xlsx processado e salvo na pasta pública.
✅ relatorios_turmas_diagnostica_18082025.xlsx processado e salvo na pasta pública.
✅ relatorios_turmas_diagnostica_19082025.xlsx processado e salvo na pasta pública.
✅ relatorios_tur